In [113]:
import pandas as pd

In [114]:
# Open up the data
filepath = '../data/wadhwani/FINAL (WIP)-MCA Job list August 2025.xlsx'
final_jobs_df = pd.read_excel(filepath)
sector_reports_jobs_df = pd.read_excel(filepath, sheet_name='Pass 3 - Sector reports', dtype={'PSOC Code': str})

# Clean up the job titles
final_jobs_df['Job Title'] = final_jobs_df['Job Title'].str.strip()
sector_reports_jobs_df['Job Title'] = sector_reports_jobs_df['Job Title'].str.strip()

# 1. Mapping MCA jobs to PSOC jobs

There are two sheets of interest for us for the Excel file *FINAL (WIP)-MCA Job list August 2025.xlsx*. Namely, they are **Final** and **Pass 3 - Sector reports**. The former is important because it contains the final list of jobs that should have the orderer pair of the exposure v complementarity scores. The latter is an auxilliary sheet that contains the PSOC codes of jobs. What we noticed is all jobs that appear in the Final job set also appear in the Pass 3 - Sector reports job set.

In [115]:
final_jobs_set = set(final_jobs_df['Job Title'])
sector_reports_jobs_df_set = set(sector_reports_jobs_df['Job Title'])

intersection = final_jobs_set.intersection(sector_reports_jobs_df_set)
union = final_jobs_set.union(sector_reports_jobs_df_set)

remaining_jobs = final_jobs_set - sector_reports_jobs_df_set

print(f'There are {len(remaining_jobs)}'
      f' ({round(len(remaining_jobs) / len(final_jobs_set), 2) * 100}%)'
      ' of jobs that are present in the Final job set but are not in the' 
      ' Pass 3 - Sector reports job set.')

There are 0 (0.0%) of jobs that are present in the Final job set but are not in the Pass 3 - Sector reports job set.


In [116]:
# define the psoc mapping already present
psoc_mapping = dict(zip(sector_reports_jobs_df['Job Title'], sector_reports_jobs_df['PSOC Code']))

# Add in the final jobs the PSOC
final_jobs_df['PSOC Code'] = final_jobs_df['Job Title'].map(psoc_mapping)

# 2. Mapping PSOC Codes to ISCO Codes

The data is available from https://psa.gov.ph/classification/psoc

In [117]:
# Get the data
filename = '../data/labor_codes/2022-Updates-to-the-2012-PSOC.xlsx'
relevant_cols = [2, 5]
names = ['PSOC', 'ISCO']
df_maps = pd.read_excel(
    filename, 
    usecols=relevant_cols, 
    names=names,
    sheet_name=None,
    dtype={"PSOC": str, "ISCO": str}
    )

# for each df_map in df_maps, get their jobs to ISCO pairs
psoc_isco = {}
for _, df_map in df_maps.items():
    df_map.dropna(inplace=True)
    mapping = dict(zip(df_map['PSOC'], df_map['ISCO']))
    psoc_isco.update(mapping)

# final_jobs_df['ISCO Code'] = final_jobs_df['PSOC Code'].map(psoc_isco)

When converting PSOC codes to ISCO codes, we identified several data-entry errors and instances where non-existent PSOC codes had been used. In these cases, we corrected the PSOC code where an appropriate code could be identified or used the closest applicable PSOC classification based on the job title.

* Embassy Administrative Attaché, Ministry Program Assistant, Co-op Administrative Support, and Seminary Support Staff: Recorded as PSOC 4111, which does not exist. The correct code is 4110 (General Office Clerk).
* Livestock Agriculture Technician: Recorded as PSOC 3144, which corresponds to Air Traffic Controllers. The closest applicable code is 2132 (Farming, Forestry and Fisheries Advisers).
* Electric Vehicle Mechanics and Repairers: PSOC 7414 has no direct ISCO mapping. The closest applicable classification is 7412 (Electrical Mechanics and Fitters).
* Chemical Sprayer: Recorded as PSOC 6110, which does not exist. The closest applicable classification is 6121 (Livestock Farmer).

In [118]:
# Correct the wrong PSOC codes
corrected_psoc_codes = {
    '4111' : '4110',
    '3144' : '2132',
    '7414' : '7412',
    '6110' : '6121'
}
final_jobs_df['PSOC Code'] = final_jobs_df['PSOC Code'].replace(corrected_psoc_codes)

# Convert from PSOC to ISCO
final_jobs_df['ISCO Code'] = final_jobs_df['PSOC Code'].map(psoc_isco)
# Clean up the ISCO codes by just only keeping the digits
final_jobs_df['ISCO Code'] = final_jobs_df['ISCO Code'].str.extract(r'(\d+)', expand=False)

# 3. Mapping ISCO Codes to 2018 O*NET (SOC) Codes

There was a data entry error in the PSA where for 6127 (HOG RAISING PRODUCERS) where the ISCO code is just 612. This is an invalid ISCO code. It should be 6121 (Livestock and dairy producers), so we will just fix that part.

However, the bigger problem is that one ISCO code can be mapped into may 2010 SOC codes. The crosswalk is found here https://www.bls.gov/soc/soccrosswalks.htm. Then we will have another crosswalk from the 2010 SOC codes to the 2018 SOC codes. The data for that is found from https://www.bls.gov/soc/2018/crosswalks.htm

In [119]:
# BUILD THE MACHINERY FOR THE ISCO TO 2010 SOC crosswalk
filename = '../data/labor_codes/ISCO_SOC_Crosswalk.xls'
isco_soc_df = pd.read_excel(filename,
                            usecols=[0, 3, 4],
                            names=['ISCO', 'SOC', 'SOC_Title'],
                            skiprows=6,
                            dtype={'ISCO': str, 'SOC': str, 'SOC_Title': str})
isco_soc_df['SOC'] = isco_soc_df['SOC'].str.strip()

# create the mappings for isco -> 2008 soc code
isco_soc = (
    isco_soc_df
    .groupby('ISCO')['SOC']
    .apply(list)
    .to_dict()
)

# fix the error
final_jobs_df['ISCO Code'] = final_jobs_df['ISCO Code'].replace({'612' : '6121'})

In [120]:
# BUILD THE MACHINERY FOR 2010 SOC to 2018 SOC crosswalk
def update_2010_2018_SOC_Codes(codes):
    """
        Given a list of codes from the 2010 SOC codes, update to the new 2018
    """
    new_codes = []
    for code in codes:
        new_codes.append(soc_2010_2018_map[code])
    return new_codes

# create the mapping form 2008 to 2018 SOC code
filename = '../data/labor_codes/soc_2010_to_2018_crosswalk.xlsx'
soc_2010_2018_df = pd.read_excel(
    filename,
    usecols=[0, 2],
    names =[2010, 2018],
    dtype={2010:'str', 2018:'str'},
    skiprows=9
)
soc_2010_2018_map = dict(zip(soc_2010_2018_df[2010], soc_2010_2018_df[2018]))

In [121]:
# convert the isco code to soc code
final_jobs_df['SOC Codes'] = final_jobs_df['ISCO Code'].map(isco_soc)

# update the 2010 codes to 2018
final_jobs_df['SOC Codes'] = final_jobs_df['SOC Codes'].apply(update_2010_2018_SOC_Codes)

# 5. Map from 2018 O*NET SOC Codes to 2019 O\*NET SOC Codes

Data is found from https://www.onetcenter.org/taxonomy/2019/soc.html

In [122]:
# create the machinery for the crosswalk from 2018 to 2019
crosswalk_2018_2019_df = pd.read_csv('../data/labor_codes/2019_to_SOC_Crosswalk.csv')
crosswalk_2018_2019_map = {code : [] for code in crosswalk_2018_2019_df['2018 SOC Code']}

for _, code_crosswalk in crosswalk_2010_2018_df.iterrows():
    code_2019 = code_crosswalk['O*NET-SOC 2019 Code']
    code_2018 = code_crosswalk['2018 SOC Code']
    crosswalk_2018_2019_map[code_2018].append(code_2019)

def convert_codes(codes, crosswalk_map):
    """Convert a list of codes using a crosswalk dictionary."""
    converted_codes = []
    for code in codes:
        converted_codes.extend(crosswalk_map[code])
    return converted_codes

In [123]:
# convert from 2018 to 2019
final_jobs_df['SOC Codes'] = final_jobs_df['SOC Codes'].apply(
    convert_codes,
    crosswalk_map=crosswalk_2018_2019_map
)

# 6. Saving Time

In [124]:
# save this 
final_jobs_df.to_csv('../data/auxiliary/mca_soc_codes.csv', index=False)